# Moduł 4: Request Body & Response Models

**Ćwiczenie:** #4 - Request Body + Response Models

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Kluczowe koncepcje](#2-kluczowe-koncepcje)
   - [2.1. Request Body - Podstawy](#21-request-body---podstawy)
   - [2.2. Multiple Body Parameters](#22-multiple-body-parameters)
   - [2.3. Body + Path + Query Parameters](#23-body--path--query-parameters-mix)
   - [2.4. Response Model - Wymuszenie struktury](#24-response-model---wymuszenie-struktury)
   - [2.5. Status Codes w Response Model](#25-status-codes-w-response-model)
   - [2.6. List Response Model](#26-list-response-model)
   - [2.7. Optional Response (Union types)](#27-optional-response-union-types)
   - [2.8. Response Model z Excludes/Includes](#28-response-model-z-excludesincludes)
   - [2.9. Form Data vs JSON](#29-form-data-vs-json)
3. [Best Practices](#3-best-practices)
4. [Demo Live Coding - Contact Form API](#4-demo-live-coding)
5. [Przygotowanie do ćwiczenia](#5-przygotowanie-do-ćwiczenia)
6. [Dodatkowe materiały](#6-dodatkowe-materiały)

---

## 1. Wprowadzenie

### Po co ten temat?

To połączenie wszystkiego czego nauczyliśmy się do tej pory:
- **HTTP methods** (POST, PUT, PATCH) wymagają body
- **Pydantic** waliduje i serializuje ten body
- **Response models** wymuszają strukturę i dokumentują API

W tym module skupimy się na **praktycznym użyciu** request/response w realnych scenariuszach.

### Gdzie to użyjemy w praktyce?

- **Formularz kontaktowy** - POST z body (imię, email, wiadomość)
- **Rejestracja użytkownika** - POST z walidacją
- **Aktualizacja profilu** - PUT/PATCH z partial updates
- **API dla aplikacji mobilnej** - wymuszenie struktury odpowiedzi

**Dlaczego to ważne?**
- 90% API wymaga request body (create, update operations)
- Response models = gwarancja struktury dla klientów (frontend, mobile)
- Automatyczna dokumentacja = mniej pytań od innych devów

---

## 2. Kluczowe koncepcje

### 2.1. Request Body - Podstawy

**Jak to działa:**
1. Klient wysyła JSON w body requestu
2. FastAPI parsuje JSON → Pydantic model
3. Pydantic waliduje dane
4. Jeśli OK → funkcja dostaje zwalidowany obiekt
5. Jeśli błąd → automatyczny 422 z opisem błędów

**Najprostszy przykład:**

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Message(BaseModel):
    text: str
    priority: int

@app.post("/messages")
def create_message(message: Message):
    """
    Stwórz wiadomość

    FastAPI automatycznie:
    - Parsuje JSON z body
    - Waliduje według Message schema
    - Zwraca 422 jeśli validation error
    """
    return {
        "received": message.text,
        "priority": message.priority
    }

**Request:**
```bash
curl -X POST http://localhost:8000/messages \
  -H "Content-Type: application/json" \
  -d '{"text": "Hello World", "priority": 1}'

# → {"received": "Hello World", "priority": 1}
```

---

### 2.2. Multiple Body Parameters

**Możesz przyjąć więcej niż jeden model:**

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    email: str

class Message(BaseModel):
    subject: str
    body: str

@app.post("/send-email")
def send_email(user: User, message: Message):
    """
    Wyślij email

    Body JSON:
    {
      "user": {...},
      "message": {...}
    }
    """
    return {
        "to": user.email,
        "subject": message.subject,
        "body": message.body
    }

**Request:**
```bash
curl -X POST http://localhost:8000/send-email \
  -H "Content-Type: application/json" \
  -d '{
    "user": {
      "name": "John",
      "email": "john@example.com"
    },
    "message": {
      "subject": "Hello",
      "body": "Test message"
    }
  }'
```

**FastAPI automatycznie** rozpoznaje że chcesz nested structure z kluczami `user` i `message`.

---

### 2.3. Body + Path + Query Parameters (Mix)

**Możesz łączyć wszystkie typy parametrów:**

In [ ]:
from pydantic import BaseModel

class PostUpdate(BaseModel):
    title: str
    content: str

@app.put("/posts/{post_id}")
def update_post(
    post_id: int,                # PATH parameter
    post: PostUpdate,            # BODY
    published: bool = False      # QUERY parameter
):
    """
    Zaktualizuj post

    URL: /posts/42?published=true
    Body: {"title": "...", "content": "..."}
    """
    return {
        "post_id": post_id,
        "title": post.title,
        "content": post.content,
        "published": published
    }

**Request:**
```bash
curl -X PUT "http://localhost:8000/posts/42?published=true" \
  -H "Content-Type: application/json" \
  -d '{"title": "Updated Title", "content": "Updated content"}'
```

**Zasada:**
- **Path params** - w URL path (`{post_id}`)
- **Query params** - po `?` w URL (`?published=true`)
- **Body** - w JSON body requestu

---

### 2.4. Response Model - Wymuszenie struktury

**Dlaczego używać `response_model`?**

1. **Dokumentacja** - Swagger pokazuje dokładny format odpowiedzi
2. **Filtrowanie** - Usuwa pola nie zdefiniowane w schema (np. password)
3. **Walidacja** - Wykrywa błędy w twoim kodzie (zwracasz zły typ)
4. **Type safety** - IDE wie co zwracasz

**Przykład:**

In [ ]:
from pydantic import BaseModel

class UserIn(BaseModel):
    """Request - Z password"""
    username: str
    email: str
    password: str

class UserOut(BaseModel):
    """Response - BEZ password"""
    id: int
    username: str
    email: str

@app.post("/users", response_model=UserOut)
def create_user(user: UserIn):
    """
    Stwórz użytkownika

    - Input: UserIn (z password)
    - Output: UserOut (BEZ password)
    """
    # Simulacja: zapis do DB, hashowanie password
    return {
        "id": 123,
        "username": user.username,
        "email": user.email,
        "password": user.password  # To zostanie ZIGNOROWANE!
    }

**Response:**
```json
{
  "id": 123,
  "username": "johndoe",
  "email": "john@example.com"
}
```

**Zauważ:** Password jest **automatycznie filtrowane** mimo że zwracamy go w dict!

---

### 2.5. Status Codes w Response Model

**Łączenie response_model z status_code:**

In [ ]:
from fastapi import status

@app.post(
    "/users",
    response_model=UserOut,
    status_code=status.HTTP_201_CREATED
)
def create_user(user: UserIn):
    """
    Stwórz użytkownika

    - Status: 201 Created
    - Response: UserOut schema
    """
    return {...}

**Best practice:**
- POST (create) → **201 Created** + response_model
- GET (read) → **200 OK** + response_model
- PUT/PATCH (update) → **200 OK** + response_model
- DELETE → **204 No Content** (bez response_model!)

---

### 2.6. List Response Model

**Zwracanie listy obiektów:**

In [ ]:
from pydantic import BaseModel

class Post(BaseModel):
    id: int
    title: str
    author: str

@app.get("/posts", response_model=list[Post])
def get_posts():
    """
    Pobierz listę postów

    Response: Lista obiektów Post
    """
    return [
        {"id": 1, "title": "Post 1", "author": "John"},
        {"id": 2, "title": "Post 2", "author": "Jane"},
    ]

**Response:**
```json
[
  {"id": 1, "title": "Post 1", "author": "John"},
  {"id": 2, "title": "Post 2", "author": "Jane"}
]
```

**FastAPI waliduje** że każdy element listy ma strukturę `Post`.

---

### 2.7. Optional Response (Union types)

**Czasem endpoint może zwrócić różne typy:**

In [ ]:
from typing import Union
from pydantic import BaseModel

class Success(BaseModel):
    message: str
    data: dict

class Error(BaseModel):
    error: str
    code: int

@app.get("/data/{id}", response_model=Union[Success, Error])
def get_data(id: int):
    """
    Pobierz dane - może zwrócić Success lub Error
    """
    if id > 0:
        return Success(message="OK", data={"id": id})
    else:
        return Error(error="Invalid ID", code=400)

**Uwaga:** Przykład czysto teoretyczny, w praktyce nigdy nie realizowany - lepiej używać HTTPException dla błędów.

---

### 2.8. Response Model z Excludes/Includes

**Kontrola jakie pola zwrócić:**

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    username: str
    email: str
    password: str
    is_active: bool

# Opcja 1: response_model_exclude
@app.get("/users/{id}", response_model=User, response_model_exclude={"password"})
def get_user(id: int):
    """Zwróć user BEZ password"""
    return {
        "id": id,
        "username": "john",
        "email": "john@example.com",
        "password": "hashed_pass",  # Zostanie pominięte
        "is_active": True
    }

# Opcja 2: response_model_include
@app.get("/users/{id}/basic", response_model=User, response_model_include={"id", "username"})
def get_user_basic(id: int):
    """Zwróć TYLKO id i username"""
    return {
        "id": id,
        "username": "john",
        "email": "john@example.com",  # Zostanie pominięte
        "password": "hashed_pass",    # Zostanie pominięte
        "is_active": True              # Zostanie pominięte
    }

**Uwaga:** Lepiej stworzyć osobne schema (`UserBasic`) niż używać exclude/include!

---

### 2.9. Form Data vs JSON

**JSON (default):**

In [ ]:
from pydantic import BaseModel

class LoginData(BaseModel):
    username: str
    password: str

@app.post("/login")
def login(data: LoginData):
    """
    Accept JSON:
    Content-Type: application/json
    Body: {"username": "john", "password": "pass"}
    """
    return {"token": "abc123"}

**Form Data (HTML forms):**

In [ ]:
from fastapi import Form

@app.post("/login")
def login(
    username: str = Form(...),
    password: str = Form(...)
):
    """
    Accept form data:
    Content-Type: application/x-www-form-urlencoded
    Body: username=john&password=pass
    """
    return {"token": "abc123"}

**Kiedy używać:**
- **JSON** - REST API, mobile apps, SPA (99% przypadków)
- **Form** - Klasyczne HTML forms, OAuth flows

---

## 3. Best Practices

### ✅ Rób tak:

**1. Zawsze używaj response_model:**

In [ ]:
# ✅ Dobre - jasna struktura odpowiedzi
@app.post("/users", response_model=UserOut)
def create_user(user: UserIn):
    return {...}

# ❌ Złe - niejasna struktura
@app.post("/users")
def create_user(user: UserIn):
    return {...}  # Co zwraca? Klient nie wie!

**2. Rozdziel Input i Output schemas:**

In [ ]:
# ✅ Dobre
class UserCreate(BaseModel):  # Input - z password
    username: str
    password: str

class UserResponse(BaseModel):  # Output - bez password
    id: int
    username: str

@app.post("/users", response_model=UserResponse)
def create_user(user: UserCreate):
    return {...}

**3. Używaj właściwych status codes:**

In [ ]:
# ✅ Dobre
@app.post("/users", status_code=201)  # Created
def create_user(user: UserCreate):
    return {...}

@app.get("/users/{id}", status_code=200)  # OK
def get_user(id: int):
    return {...}

@app.delete("/users/{id}", status_code=204)  # No Content
def delete_user(id: int):
    return  # Brak body!

**4. Dodawaj descriptions do schemas:**

In [ ]:
from pydantic import Field

# ✅ Dobre
class UserCreate(BaseModel):
    """Schema for creating a new user"""
    username: str = Field(description="Unique username")
    email: str = Field(description="Valid email address")
    password: str = Field(description="Strong password (min 8 chars)")

**5. Grupuj powiązane pola w nested models:**

In [ ]:
# ✅ Dobre - czytelne i reusable
class Address(BaseModel):
    street: str
    city: str

class UserCreate(BaseModel):
    name: str
    address: Address

# ❌ Złe - płaska struktura
class UserCreate(BaseModel):
    name: str
    street: str
    city: str

---

### ❌ Nie rób tak:

**1. Nie zwracaj password w response:**

In [ ]:
# ❌ Złe - BARDZO ZŁE!
class User(BaseModel):
    username: str
    password: str  # Nigdy nie zwracaj password!

@app.get("/users/{id}", response_model=User)
def get_user(id: int):
    return {...}

# ✅ Dobre - osobny schema bez password
class UserResponse(BaseModel):
    username: str

@app.get("/users/{id}", response_model=UserResponse)
def get_user(id: int):
    return {...}

**2. Nie używaj dict zamiast Pydantic model:**

In [ ]:
# ❌ Złe - brak walidacji
@app.post("/users")
def create_user(user: dict):
    # dict może zawierać cokolwiek!
    return user

# ✅ Dobre - walidacja
@app.post("/users")
def create_user(user: UserCreate):
    return user

**3. Nie ignoruj validation errors:**

In [ ]:
# ❌ Złe - ręczna walidacja
@app.post("/users")
def create_user(user: UserCreate):
    if not user.email:
        return {"error": "Email required"}  # Niepotrzebne!

# ✅ Dobre - Pydantic waliduje
class UserCreate(BaseModel):
    email: str  # Wymagane pole - Pydantic sprawdzi

@app.post("/users")
def create_user(user: UserCreate):
    # Email już zwalidowany
    return {...}

**4. Nie mieszaj JSON i Form w jednym endpoincie:**

In [ ]:
# ❌ Złe - nieokreślone
@app.post("/users")
def create_user(user: UserCreate, username: str = Form(...)):
    # Co klient ma wysłać - JSON czy Form?
    pass

# ✅ Dobre - jeden format
@app.post("/users")
def create_user(user: UserCreate):  # Tylko JSON
    pass

**5. Nie zwracaj body dla 204:**

In [ ]:
# ❌ Złe - 204 nie powinno mieć body
@app.delete("/users/{id}", status_code=204, response_model=MessageResponse)
def delete_user(id: int):
    return {"message": "Deleted"}  # Konflikt!

# ✅ Dobre - brak body dla 204
@app.delete("/users/{id}", status_code=204)
def delete_user(id: int):
    # Usuń z DB
    return  # Brak body

---

## 4. Demo Live Coding

### Przykład: Contact Form API

**Krok 1: Setup**

In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, EmailStr, ConfigDict
from typing import List, Optional
from datetime import datetime

app = FastAPI(title="Contact Form API")

# Mock database
messages_db = []

**Krok 2: Request schemas**

In [ ]:
class ContactMessage(BaseModel):
    """Schema for submitting contact form"""
    name: str = Field(min_length=2, max_length=100, description="Sender's name")
    email: EmailStr = Field(description="Valid email address")
    subject: str = Field(min_length=5, max_length=200, description="Message subject")
    message: str = Field(min_length=10, max_length=5000, description="Message content")
    phone: Optional[str] = Field(None, description="Optional phone number")

    model_config = ConfigDict(
        json_schema_extra={
            "examples": [
                {
                    "name": "John Doe",
                    "email": "john@example.com",
                    "subject": "Question about your service",
                    "message": "I would like to know more about...",
                    "phone": "+48 123 456 789"
                }
            ]
        }
    )

**Krok 3: Response schemas**

In [ ]:
class MessageResponse(BaseModel):
    """Schema for response after submission"""
    id: int
    name: str
    email: EmailStr
    subject: str
    submitted_at: datetime
    status: str  # "pending", "processed"

class MessageListResponse(BaseModel):
    """Schema for list of messages"""
    total: int
    messages: List[MessageResponse]

**Krok 4: POST endpoint (submit form)**

In [ ]:
@app.post(
    "/contact",
    response_model=MessageResponse,
    status_code=status.HTTP_201_CREATED,
    tags=["contact"]
)
def submit_contact_form(message: ContactMessage):
    """
    Submit contact form

    **Walidacja:**
    - Name: 2-100 znaków
    - Email: poprawny format
    - Subject: 5-200 znaków
    - Message: 10-5000 znaków
    - Phone: opcjonalny

    **Response:** 201 Created z ID wiadomości
    """
    # Simulacja zapisu do DB
    new_message = {
        "id": len(messages_db) + 1,
        "name": message.name,
        "email": message.email,
        "subject": message.subject,
        "message": message.message,
        "phone": message.phone,
        "submitted_at": datetime.now(),
        "status": "pending"
    }

    messages_db.append(new_message)

    # Response nie zawiera pełnej treści wiadomości (message field)
    # bo MessageResponse nie ma tego pola
    return new_message

**Test (sukces):**
```bash
curl -X POST http://localhost:8000/contact \
  -H "Content-Type: application/json" \
  -d '{
    "name": "John Doe",
    "email": "john@example.com",
    "subject": "Test Subject",
    "message": "This is a test message with more than 10 characters."
  }'

# → 201 Created
# {
#   "id": 1,
#   "name": "John Doe",
#   "email": "john@example.com",
#   "subject": "Test Subject",
#   "submitted_at": "2024-01-15T10:30:00",
#   "status": "pending"
# }
```

**Test (validation error):**
```bash
# Message zbyt krótki
curl -X POST http://localhost:8000/contact \
  -H "Content-Type: application/json" \
  -d '{
    "name": "John",
    "email": "john@example.com",
    "subject": "Test",
    "message": "Short"
  }'

# → 422 Unprocessable Entity
# {
#   "detail": [
#     {
#       "loc": ["body", "message"],
#       "msg": "ensure this value has at least 10 characters",
#       "type": "value_error.any_str.min_length"
#     }
#   ]
# }
```

**Krok 5: GET endpoint (lista wiadomości)**

In [ ]:
@app.get(
    "/contact",
    response_model=MessageListResponse,
    tags=["contact"]
)
def get_messages(status_filter: Optional[str] = None):
    """
    Pobierz listę wiadomości

    - **status_filter**: Filtruj po statusie (pending/processed)
    """
    messages = messages_db

    if status_filter:
        messages = [m for m in messages if m["status"] == status_filter]

    return {
        "total": len(messages),
        "messages": messages
    }

**Test:**
```bash
curl http://localhost:8000/contact
# → {
#     "total": 2,
#     "messages": [...]
#   }

curl "http://localhost:8000/contact?status_filter=pending"
# → Tylko pending messages
```

**Krok 6: GET pojedyncza wiadomość (z pełną treścią)**

In [ ]:
class MessageDetailResponse(BaseModel):
    """Pełne szczegóły wiadomości (z message content)"""
    id: int
    name: str
    email: EmailStr
    subject: str
    message: str  # Full message content
    phone: Optional[str]
    submitted_at: datetime
    status: str

@app.get(
    "/contact/{message_id}",
    response_model=MessageDetailResponse,
    tags=["contact"]
)
def get_message(message_id: int):
    """
    Pobierz szczegóły wiadomości

    - **message_id**: ID wiadomości
    """
    message = next((m for m in messages_db if m["id"] == message_id), None)

    if not message:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Message {message_id} not found"
        )

    return message

**Test:**
```bash
curl http://localhost:8000/contact/1
# → Pełne szczegóły wiadomości 1

curl http://localhost:8000/contact/999
# → 404 Not Found
```

---

## Podsumowanie

### Kluczowe zagadnienia:

1. **Request Body** - Pydantic models automatycznie walidują JSON body
2. **Response Model** - Wymusza strukturę, filtruje wrażliwe dane, dokumentuje API
3. **Multiple Schemas** - Rozdzielaj Input (UserCreate) i Output (UserResponse)
4. **Status Codes** - 201 (Created), 200 (OK), 204 (No Content)
5. **Nested Models** - Grupuj powiązane pola (Address w User)
6. **List[Model]** - Zwracaj listy typowanych obiektów
7. **Best Practices** - Zawsze używaj response_model, rozdziel schemas, właściwe kody statusu
